In [ ]:
from minimal_lc0_for_research.leela_board import LeelaBoard

board = LeelaBoard()
board

In [ ]:
import chess
import numpy as np
board.push_uci('e2e4')
b1 = board.lcz_features()
b2 = LeelaBoard(fen='8/rnbqkbnr/pppppppp/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1')
dif = b1-b2.lcz_features()
print(np.sum(dif, axis=None))

4192


In [8]:
!zstd -d  lichess_db_puzzle.csv.zst

lichess_db_puzzle.csv.zst: 1059865039 bytes                                    


In [4]:
target_lichess_classes = [
    'exposedKing',
    'sacrifice',
    'hangingPiece',
    'fork',
    'captureTheDefender',
    'pin',
    'quietMove',
    'intermezzo',
    'deflection'
]
additional_target_classes = [
    'planlessGame',
]
targets = target_lichess_classes+additional_target_classes

In [5]:
import os
import pandas as pd
from minimal_lc0_for_research.leela_board import LeelaBoard
try:
    os.mkdir('data')
except FileExistsError:
    pass
df = pd.read_csv('lichess_db_puzzle/part_0.csv')
print(df.shape)

(100000, 10)


In [23]:
print(df['Moves'].values[0])
b = LeelaBoard(df['FEN'].values[0])
print(b)
print(set(df))

f2g3 e6e7 b2b1 b3c1 b1c1 h6c1
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Turn: White
{'OpeningTags', 'FEN', 'PuzzleId', 'RatingDeviation', 'Moves', 'NbPlays', 'GameUrl', 'Themes', 'Popularity', 'Rating'}


In [43]:
from chess import Board
print(LeelaBoard(fen=df['FEN'][0]))
print(Board(df['FEN'][0]))

r . . . . . . k
p p . . r . . p
. . . . R p . Q
. . . p . . . .
. . . . . . . .
. N . P . . R .
P q P . . b P P
. . . . . . . K
Turn: Black
r . . . . . . k
p p . . r . . p
. . . . R p . Q
. . . p . . . .
. . . . . . . .
. N . P . . R .
P q P . . b P P
. . . . . . . K


In [2]:
from stockfish import Stockfish
path_to_binary = 'stockfish/stockfish-ubuntu-x86-64-avx2'
engine = Stockfish(path=path_to_binary, depth=1)

In [7]:
fen = df['FEN'][0]  # After 1. e4
engine.set_fen_position(fen)
eval_result = engine.get_evaluation()['value']/100
print(eval_result)

2.31


In [ ]:
'''import numpy as np
from tqdm import tqdm
from shutil import rmtree
import os
from stockfish import Stockfish

path = 'data'
if os.path.exists:
    rmtree(path)
os.mkdir('data')

def generate_target_vector(tags: str):
        
    tags = tags.split(' ')
    indices = []
    one_hot = np.zeros(len(targets), dtype=bool)
    for tag in tags:
        try:
            ind = targets.index(tag)
            one_hot[ind]=True
            break
        except ValueError:
            continue
    return one_hot

n_instances = 5000
n_classes = len(target_lichess_classes) + len(additional_target_classes)

path_to_binary = 'stockfish/stockfish-ubuntu-x86-64-avx2'
engine = Stockfish(path=path_to_binary, depth=1)

def generate_batch_instance(k: int):
    a = np.ndarray(shape=(n_instances, 5, 8, 8, 112), dtype=np.uint8)
    b = np.ndarray(shape=(n_instances, n_classes), dtype=bool)
    d = n_instances*k #Actual position in the table
    evals = np.ndarray(shape=(n_instances, 5)) #evaluations of positions
    for i in tqdm(range(n_instances)):
        FEN = cur_df['FEN'].values[i+d]
        moves_str = cur_df['Moves'].values[i+d]
        UCIs = moves_str.split(' ')[:-4] 
        engine.set_fen_position(FEN)
        evals[i][0]=(engine.get_evaluation()['value'])/100
        board = LeelaBoard(fen=FEN)
        
        a[i][0] = np.moveaxis(board.lcz_features(), 0, -1)
        
        for j in range(min(len(UCIs), 4)):
            try:
                board.push_uci(UCIs[j])
                engine.make_moves_from_current_position([UCIs[j]])
                a[i][j+1] = np.moveaxis(board.lcz_features(), 0, -1)
                evals[i][j+1] = engine.get_evaluation()['value']
            except Exception as e:
                print(f"Error at instance {i}, move {j}: {e}")
                break 

        # 2. Генерация таргета (вместо np.vectorize используем прямой вызов)
        b[i] = generate_target_vector(cur_df['Themes'].values[i+d])

    np.savez(f'data/batch{k}.npz', x=a, y=b, evals=evals)
    print(f'Batch {k} sucessfully built')
for d in range(10):
    cur_df = pd.read_csv(f'lichess_db_puzzle/part_{d}.csv')
    for i in range(0, cur_df.shape[0]//n_instances):
        generate_batch_instance(i)'''

Now we generate dataset for binary classifier
Sample random sequences of 5 moves from good player's games 

In [ ]:

!unzip data/lichess_elite_2022_01.zip

In [37]:
from chess.pgn import read_game
import numpy as np
from minimal_lc0_for_research.leela_board import LeelaBoard
from tqdm import tqdm
from stockfish import Stockfish

n_game_samples=2
class_weight = 0.1
n_moves = 5
def get_game_fens(batch_size=10):
    with open('data/lichess_elite_2022-01.pgn') as f:
        FENs = []
        all_moves = []
        for _ in range(int(batch_size*(1-class_weight)//n_game_samples)):
            game = read_game(f)
            
            moves = list(game.mainline_moves()); total = len(moves)
            av = list(range(total))
            p = np.array([True for _ in range(total)]).astype(bool)
            for _ in range(n_game_samples):
                board = game.board()
                idx = np.random.choice(av)
                while (not p[idx]): idx = np.random.choice(av)
                p[idx-n_moves:idx+n_moves]=False
                for i in range(0, idx):
                    board.push(moves[i])
                FENs.append(board.fen())
                all_moves.append(moves[idx:idx+n_moves])
    return FENs, all_moves



def generate_binary_chunk(chunksize:int=500, class_weight:float=0.1, silent:bool=True, save:bool=False):
    '''
    Creates chunk of data for binary classifier
    kwargs: 
    chunksize: number of instances of both classes
    class_weight: proportion between class 1 (positive) and class 2 (negative)
    '''
    path_to_binary = 'stockfish/stockfish-ubuntu-x86-64-avx2'
    engine = Stockfish(path=path_to_binary, depth=1)
    FENs, moves = get_game_fens(chunksize)
    if not silent: print(f"Len of FENs is {len(FENs)} len of moves is {len(moves)}")
    n_negative = int(chunksize*(1-class_weight))
    n_positive = int(chunksize*class_weight)
    negative_evals = np.zeros(shape=(n_negative, 5))
    negative_positions = np.zeros(shape=(n_negative, 5, 8, 8, 112))
    for i in tqdm(range(len(FENs))):

        engine.set_fen_position(FENs[i])
        board = LeelaBoard(fen=FENs[i])

        for j in range(n_moves):
            cur_move = moves[i][j].uci()
            board.push_uci(cur_move)
            engine.make_moves_from_current_position([cur_move])
            negative_evals[i][j]=(engine.get_evaluation()['value'])/100
            negative_positions[i][j]=np.moveaxis(board.lcz_features(), 0, -1)


    i = np.random.randint(0, 20); a = np.load(f"data/batch{i}.npz")
    idx = np.random.randint(0, a['x'].shape[0]-n_positive)
    positive_positions = a['x'][idx:idx+n_positive]
    positive_evals = a['evals'][idx:idx+n_positive]
    indices = np.random.shuffle(np.arange(chunksize))

    if not silent: print(f'positive positions shape is {positive_positions.shape}, positive evals shape is {positive_evals.shape}\
        ')

    y = np.concat([np.zeros(shape=(n_negative, 1)), np.ones(shape=(n_positive, 1))], axis=0)
    all_positions = np.concat([negative_positions, positive_positions], axis=0)
    all_evals = np.concat([negative_evals, positive_evals], axis=0)
    y = y[indices]
    all_positions = all_positions[indices]
    all_evals = all_evals[indices]
    if save:
        filename = "BinaryClassifierData/test.npz"
        np.savez(file=filename, x=all_positions, y = y, evals=all_evals)
        print("Successfully saved test.npz")
    else:
        return (all_positions, y, all_evals)
    

In [38]:
generate_binary_chunk(chunksize=10, silent=False, save=True)

Len of FENs is 8 len of moves is 8


100%|██████████| 8/8 [00:00<00:00, 109.04it/s]


positive positions shape is (1, 5, 8, 8, 112), positive evals shape is (1, 5)        
Successfully saved test.npz
